In [ ]:
"""
LICENSE MIT
2021
Guillaume Rozier
Website : http://www.covidtracker.fr
Mail : guillaume.rozier@telecomnancy.net

README:
This file contains scripts that download data from data.gouv.fr and then process it to build many graphes.
I'm currently cleaning the code, please ask me if something is not clear enough.

The charts are exported to 'charts/images/france'.
Data is download to/imported from 'data/france'.
Requirements: please see the imports below (use pip3 to install them).

"""

In [1]:
import pandas as pd
import json
import france_data_management as data

show_charts = False
PATH_STATS = "../../data/france/stats/"

In [2]:
df, df_confirmed, dates, df_new, df_tests, df_deconf, df_sursaud, df_incid, df_tests_viros = data.import_data()

  0%|          | 0/8 [00:00<?, ?it/s]/Users/guillaumerozier/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3249: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):
36it [04:38,  8.30s/it]                      

In [11]:
df_incid_fra_clage = data.import_data_tests_sexe()
df_incid_fra = df_incid_fra_clage[df_incid_fra_clage["cl_age90"]==0]
df_france = df.groupby(["jour"]).sum().reset_index()

In [36]:
departements = departements = list(dict.fromkeys(list(df_incid['dep'].values))) 
regions = departements = list(dict.fromkeys(list(df_incid['regionName'].values))) 

df_regions = df.groupby(["jour", "regionName"]).sum().reset_index()
df_incid_regions = df_incid.groupby(["jour", "regionName"]).sum().reset_index()

In [18]:
def generate_data(data_incid, data_hosp):## Incidence
    dict_data = {}

    taux_incidence = data_incid["P"].rolling(window=7).sum().fillna(0) * 100000/67000000
    dict_data["incidence"] = {"jour": list(data_incid.jour), "valeur": list(taux_incidence)}

    cas = data_incid["P"].rolling(window=7).mean().fillna(0)
    dict_data["cas"] = {"jour": list(data_incid.jour), "valeur": list(cas)}

    hospitalisations = data_hosp.hosp.fillna(0)
    dict_data["hospitalisations"] = {"jour": list(data_hosp.jour), "valeur": list(hospitalisations)}

    reanimations = data_hosp.rea.fillna(0)
    dict_data["reanimations"] = {"jour": list(data_hosp.jour), "valeur": list(reanimations)}

    deces_hospitaliers = data_hosp.dc.diff().rolling(window=7).mean().fillna(0)
    dict_data["deces_hospitaliers"] = {"jour": list(data_hosp.jour), "valeur": list(deces_hospitaliers)}
    
    return dict_data
 

In [19]:
def export_data(data):
    with open(PATH_STATS + 'dataexplorer.json', 'w') as outfile:
        json.dump(data, outfile)

In [40]:
def dataexplorer():
    dict_data = {}
    dict_data["france"] = generate_data(df_incid_fra, df_france)
    
    for reg in regions:
        dict_data[reg] = generate_data(df_incid_regions[df_incid_regions.regionName==reg], df_regions[df_regions.regionName==reg])
    
    export_data(dict_data)

In [41]:
dataexplorer()